In [1]:
!pip install duckdb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 1.8 MB/s eta 0:00:0000:0100:01


In [3]:
import duckdb
from pathlib import Path

CLEAN = Path.home() / "Feirando" / "clean"
con = duckdb.connect()

zip_types = {
    "customers":   {"customer_zip_code_prefix": "VARCHAR"},
    "sellers":     {"seller_zip_code_prefix": "VARCHAR"},
    "geolocation": {"geolocation_zip_code_prefix": "VARCHAR"},
}

tables = ["orders", "order_items", "order_reviews", "order_payments", "customers",
          "geolocation", "products", "sellers", "category_translation"]

for t in tables:
    path = CLEAN / f"{t}_clean.csv"
    opts = f", types={zip_types[t]}" if t in zip_types else ""
    con.execute(f"CREATE OR REPLACE TABLE {t} AS SELECT * FROM read_csv('{path}'{opts})")

con.sql("SELECT table_name, estimated_size FROM duckdb_tables()")

┌──────────────────────┬────────────────┐
│      table_name      │ estimated_size │
│       varchar        │     int64      │
├──────────────────────┼────────────────┤
│ category_translation │             74 │
│ customers            │          99441 │
│ geolocation          │          19010 │
│ orders               │          99441 │
│ order_items          │         112650 │
│ order_payments       │         103886 │
│ order_reviews        │          98673 │
│ products             │          32951 │
│ sellers              │           3095 │
└──────────────────────┴────────────────┘

In [17]:
con.sql("DESCRIBE orders")

┌───────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│          column_name          │ column_type │  null   │   key   │ default │  extra  │
│            varchar            │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ order_id                      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_id                   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ order_status                  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ order_purchase_timestamp      │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ order_approved_at             │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ order_delivered_carrier_date  │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ order_delivered_customer_date │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ order_estimated_delivery_date 

In [19]:
con.sql("""
SELECT * FROM orders LIMIT 5

""")

┌──────────────────────────────────┬──────────────────────────────────┬──────────────┬──────────────────────────┬─────────────────────┬──────────────────────────────┬───────────────────────────────┬───────────────────────────────┬───────────────────┬───────────┬────────────┬─────────┬──────────────┐
│             order_id             │           customer_id            │ order_status │ order_purchase_timestamp │  order_approved_at  │ order_delivered_carrier_date │ order_delivered_customer_date │ order_estimated_delivery_date │ in_delivery_scope │ in_window │ delay_days │ is_late │ multi_seller │
│             varchar              │             varchar              │   varchar    │        timestamp         │      timestamp      │          timestamp           │           timestamp           │             date              │      boolean      │  boolean  │   double   │ boolean │   boolean    │
├──────────────────────────────────┼──────────────────────────────────┼──────────────┼───────────

In [28]:
con.sql("""
    SELECT order_status,COUNT(order_id) AS Total_orders
    FROM orders
    GROUP BY order_status
    ORDER BY Total_orders DESC;
""")

┌──────────────┬──────────────┐
│ order_status │ Total_orders │
│   varchar    │    int64     │
├──────────────┼──────────────┤
│ delivered    │        96478 │
│ shipped      │         1107 │
│ canceled     │          625 │
│ unavailable  │          609 │
│ invoiced     │          314 │
│ processing   │          301 │
│ created      │            5 │
│ approved     │            2 │
└──────────────┴──────────────┘

In [32]:
con.sql("""
    SELECT COUNT(*)              AS rows,
           COUNT(price)     AS rows_with_value,
           MIN(price)       AS min_price,
           MAX(price)       AS max_price,
           ROUND(AVG(price), 1)  AS avg_price,
           MEDIAN(price)    AS median_price
    FROM order_items;
""")

┌────────┬─────────────────┬───────────┬───────────┬───────────┬──────────────┐
│  rows  │ rows_with_value │ min_price │ max_price │ avg_price │ median_price │
│ int64  │      int64      │  double   │  double   │  double   │    double    │
├────────┼─────────────────┼───────────┼───────────┼───────────┼──────────────┤
│ 112650 │          112650 │      0.85 │    6735.0 │     120.7 │        74.99 │
└────────┴─────────────────┴───────────┴───────────┴───────────┴──────────────┘

In [41]:
con.sql("""
    SELECT DATETRUNC('month', order_purchase_timestamp) AS month,
           COUNT(*) AS orders,
           ROUND(AVG(CASE WHEN is_late THEN 1 ELSE 0 END)*100,1) AS late_rate
    FROM orders
     WHERE in_delivery_scope AND in_window
    GROUP BY month
    ORDER BY month
""")

┌─────────────────────┬────────┬───────────┐
│        month        │ orders │ late_rate │
│      timestamp      │ int64  │  double   │
├─────────────────────┼────────┼───────────┤
│ 2017-01-01 00:00:00 │    750 │       2.9 │
│ 2017-02-01 00:00:00 │   1653 │       3.0 │
│ 2017-03-01 00:00:00 │   2546 │       4.6 │
│ 2017-04-01 00:00:00 │   2303 │       6.6 │
│ 2017-05-01 00:00:00 │   3545 │       3.0 │
│ 2017-06-01 00:00:00 │   3135 │       3.0 │
│ 2017-07-01 00:00:00 │   3872 │       2.8 │
│ 2017-08-01 00:00:00 │   4193 │       2.9 │
│ 2017-09-01 00:00:00 │   4150 │       4.4 │
│ 2017-10-01 00:00:00 │   4478 │       4.2 │
│ 2017-11-01 00:00:00 │   7288 │      12.4 │
│ 2017-12-01 00:00:00 │   5513 │       7.5 │
│ 2018-01-01 00:00:00 │   7069 │       5.7 │
│ 2018-02-01 00:00:00 │   6555 │      14.1 │
│ 2018-03-01 00:00:00 │   7003 │      19.0 │
│ 2018-04-01 00:00:00 │   6798 │       4.5 │
│ 2018-05-01 00:00:00 │   6749 │       6.6 │
│ 2018-06-01 00:00:00 │   6096 │       1.2 │
│ 2018-07-

In [42]:
con.sql("""
    SELECT multi_seller,
           COUNT(*) AS orders,
           ROUND(AVG(CASE WHEN is_late THEN 1 ELSE 0 END) * 100, 1) AS late_rate_pct
    FROM orders
    WHERE in_delivery_scope AND in_window
    GROUP BY multi_seller
""")

┌──────────────┬────────┬───────────────┐
│ multi_seller │ orders │ late_rate_pct │
│   boolean    │ int64  │    double     │
├──────────────┼────────┼───────────────┤
│ true         │   1272 │           1.0 │
│ false        │  94931 │           6.9 │
└──────────────┴────────┴───────────────┘

## H1: Do late orders get lower review scores

In [44]:
con.sql("""
    SELECT o.is_late, 
           COUNT(*) AS orders,
           ROUND(AVG(r.review_score), 1) AS avg_review_score
    FROM orders o
    JOIN order_reviews r ON o.order_id = r.order_id
    WHERE o.in_delivery_scope AND o.in_window
    GROUP BY o.is_late;
    """)

┌─────────┬────────┬──────────────────┐
│ is_late │ orders │ avg_review_score │
│ boolean │ int64  │      double      │
├─────────┼────────┼──────────────────┤
│ false   │  89182 │              4.3 │
│ true    │   6378 │              2.3 │
└─────────┴────────┴──────────────────┘

## H2: Does a longer delay in delivery mean a worse review score

In [ ]:
con.sql("""
SELECT 
        CASE
        WHEN delay_days <= 0 THEN '0 on time'
        WHEN delay_days <= 3 THEN '1 1-3 days'
        WHEN delay_days <=10 THEN '2 4-10 days'
        ELSE '3 +11 days'
        END AS delay_band,
        COUNT(*) AS orders,
        ROUND(AVG(r.review_score), 1) AS avg_review_score       
FROM orders o 
JOIN order_reviews r
ON o.order_id = r.order_id
WHERE o.in_delivery_scope AND o.in_window
GROUP BY delay_band
ORDER BY delay_band
""")


┌─────────────┬────────┬──────────────────┐
│ delay_band  │ orders │ avg_review_score │
│   varchar   │ int64  │      double      │
├─────────────┼────────┼──────────────────┤
│ 0 on time   │  89182 │              4.3 │
│ 1 0-3 days  │   1851 │              3.3 │
│ 2 4-10 days │   2504 │              2.0 │
│ 3 +11 days  │   2023 │              1.7 │
└─────────────┴────────┴──────────────────┘

## H3: Are late deliveries concentrated in a few seller

In [72]:
con.sql("""
    CREATE OR REPLACE TABLE seller_late AS
    SELECT oi.seller_id,
           COUNT(DISTINCT oi.order_id) AS orders,
           COUNT(DISTINCT CASE WHEN o.is_late THEN o.order_id END) AS late_orders,
           ROUND(late_orders * 100.0 / orders, 1) AS late_rate
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.order_id
    WHERE o.in_delivery_scope AND o.in_window AND NOT o.multi_seller
    GROUP BY oi.seller_id
    HAVING orders >= 30
""")

con.sql("SELECT COUNT(*) AS sellers, SUM(late_orders) AS late_orders FROM seller_late")

con.sql("""
    SELECT SUM(late_orders) AS worst_late_orders,
           ROUND(SUM(late_orders) * 100.0 / 5432, 1) AS share_pct
    FROM (
        SELECT *
        FROM seller_late
        ORDER BY late_rate DESC
        LIMIT 61
    )
""")

┌───────────────────┬───────────┐
│ worst_late_orders │ share_pct │
│      int128       │  double   │
├───────────────────┼───────────┤
│               818 │      15.1 │
└───────────────────┴───────────┘

In [73]:
con.sql("DESCRIBE sellers")

┌────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name       │ column_type │  null   │   key   │ default │  extra  │
│        varchar         │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ seller_id              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ seller_zip_code_prefix │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ seller_city            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ seller_state           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [74]:
con.sql("DESCRIBE customers")

┌──────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name        │ column_type │  null   │   key   │ default │  extra  │
│         varchar          │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ customer_id              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_unique_id       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_zip_code_prefix │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_city            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_state           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [75]:
con.sql("DESCRIBE order_items")

┌─────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name     │ column_type │  null   │   key   │ default │  extra  │
│       varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ order_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ order_item_id       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ product_id          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ seller_id           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ shipping_limit_date │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ price               │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ freight_value       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ shipping_limit_flag │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────────────┴─────────────┴─────────┴─────────┴───────

In [76]:
con.sql("DESCRIBE orders")

┌───────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│          column_name          │ column_type │  null   │   key   │ default │  extra  │
│            varchar            │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ order_id                      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_id                   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ order_status                  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ order_purchase_timestamp      │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ order_approved_at             │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ order_delivered_carrier_date  │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ order_delivered_customer_date │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ order_estimated_delivery_date 

## H4: Does late delivery correlate with cross-state orders? What is the late rate for cross-state orders vs same-state orders?

In [ ]:
con.sql("""

SELECT  CASE WHEN s.seller_state = c.customer_state THEN 'same_state'
             ELSE 'cross_state' END AS state_match,
        COUNT(DISTINCT oi.order_id) AS orders,
        COUNT(DISTINCT CASE WHEN o.is_late THEN oi.order_id END) AS late_orders,
        ROUND((late_orders / orders) * 100, 1) AS late_rate
        
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
JOIN sellers s ON oi.seller_id = s.seller_id

WHERE o.in_delivery_scope AND o.in_window AND NOT o.multi_seller
GROUP BY state_match

""")

┌─────────────┬────────┬─────────────┬───────────┐
│ state_match │ orders │ late_orders │ late_rate │
│   varchar   │ int64  │    int64    │  double   │
├─────────────┼────────┼─────────────┼───────────┤
│ same_state  │  34149 │        1559 │       4.6 │
│ cross_state │  60782 │        4959 │       8.2 │
└─────────────┴────────┴─────────────┴───────────┘

## H5: Do shoppers whose first order arrives late come back less often?

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE customer_orders AS
SELECT  o.order_id,
        o.is_late,
        c.customer_unique_id,
        o.order_purchase_timestamp

FROM orders o
JOIN customers c ON o.customer_id = c.customer_id

""")

con.sql("""
    CREATE OR REPLACE TABLE first_dates AS
    SELECT customer_unique_id,
           MIN(order_purchase_timestamp) AS first_order
    FROM customer_orders
    GROUP BY customer_unique_id
""")

con.sql("""
    CREATE OR REPLACE TABLE first_orders AS
    SELECT co.customer_unique_id,
           co.order_id,
           co.order_purchase_timestamp,
           co.is_late
    FROM customer_orders co
    JOIN first_dates fd
      ON co.customer_unique_id = fd.customer_unique_id
     AND co.order_purchase_timestamp = fd.first_order
""")

con.sql("SELECT COUNT(*) AS rows FROM first_orders")

# Clear first orders
con.sql("""
    CREATE OR REPLACE TABLE h5_shoppers AS
    SELECT fo.*
    FROM first_orders fo
    JOIN (
        SELECT customer_unique_id
        FROM first_orders
        GROUP BY customer_unique_id
        HAVING COUNT(*) = 1
    ) single USING (customer_unique_id)
    WHERE fo.is_late IS NOT NULL
      AND fo.order_purchase_timestamp >= '2017-01-01'
      AND fo.order_purchase_timestamp <  '2018-03-01'
""")

# did they come back within 180 days?
con.sql("""
    CREATE OR REPLACE TABLE h5_result AS
    SELECT h.customer_unique_id,
           h.is_late,
           EXISTS (
               SELECT 1
               FROM customer_orders co
               WHERE co.customer_unique_id = h.customer_unique_id
                 AND co.order_purchase_timestamp >  h.order_purchase_timestamp
                 AND co.order_purchase_timestamp <= h.order_purchase_timestamp + INTERVAL 180 DAY
           ) AS repeated
    FROM h5_shoppers h
""")

# compare the two groups
con.sql("""
    SELECT is_late AS first_order_late,
           COUNT(*) AS shoppers,
           SUM(CASE WHEN repeated THEN 1 ELSE 0 END) AS repeaters,
           ROUND(AVG(CASE WHEN repeated THEN 1 ELSE 0 END) * 100, 2) AS repeat_rate_pct
    FROM h5_result
    GROUP BY is_late
""")

┌──────────────────┬──────────┬───────────┬─────────────────┐
│ first_order_late │ shoppers │ repeaters │ repeat_rate_pct │
│     boolean      │  int64   │  int128   │     double      │
├──────────────────┼──────────┼───────────┼─────────────────┤
│ true             │     3655 │        89 │            2.44 │
│ false            │    51350 │      1475 │            2.87 │
└──────────────────┴──────────┴───────────┴─────────────────┘